# Week 12 - Sentiment Analysis (1/2)
**Notebook 1 : Problem Definition and the Lexicon-based Approach**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

In Week 10 a classifier read a row of counts and returned a news category. Sentiment analysis asks a narrower question
about a text: is the opinion in it **positive or negative**? This notebook covers lecture 10-2 *Sentiment Analysis Overview*
and chapter 3 of lecture 11-1 / 11-2 *Sentiment Analysis* (below simply "lecture 11"), on 7,500 hotel reviews.

| Lecture | What we do in code |
|---|---|
| 10-2, ch. 2 Architecture | an opinion as a quintuple, and document-level sentiment from star ratings |
| 11, ch. 3 Dictionary-based approach | count opinion words with a public lexicon, evaluate it, look at its mistakes, grow a lexicon from WordNet |
| 11, ch. 3 Corpus-based approach | build our own sentiment dictionary from the ratings with a t-test, and find domain-dependent words |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. Run the **Setup** cells first. The **🤔 Quick check** notes after an output say what to look for in it.
> The cell marked **🔧** asks you to change one value in the code and run the cell again.

## &nbsp;0. Setup
NumPy, pandas, SciPy, scikit-learn and NLTK are pre-installed in Colab. NLTK downloads four small resources:
the **opinion lexicon** of Hu and Liu (lecture 11 p. 13), **WordNet** with its helper `omw-1.4`, and **SentiWordNet** (p. 11-12).

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score

import nltk
for resource in ['opinion_lexicon', 'wordnet', 'omw-1.4', 'sentiwordnet']:
    nltk.download(resource, quiet=True)
from nltk.corpus import opinion_lexicon, wordnet as wn, sentiwordnet as swn

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 100)

**The data: TripAdvisor hotel reviews.** 20,491 reviews with a rating from 1 to 5 stars, collected by Alam et al. (2016)
and shared on Kaggle. We use a sample of **6,000 training and 1,500 test reviews**. The authors already lowercased the text
and removed most stop words, so a review reads like a list of short phrases separated by commas
(we only wrote the split contraction *n't* back as *not*).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week12-sentiment-analysis/data/tripadvisor_sample.csv"

reviews = pd.read_csv(DATA_URL)
train = reviews[reviews['split'] == 'train'].reset_index(drop=True)
test = reviews[reviews['split'] == 'test'].reset_index(drop=True)

print('train :', len(train), '  test :', len(test))
print('reviews per rating (train) :', train['rating'].value_counts().sort_index().to_dict())
print()
print(train.loc[0, 'rating'], 'stars :', train.loc[0, 'text'][:300], '...')

## &nbsp;1. What sentiment analysis turns a text into
### &nbsp;1-1. An opinion as a quintuple
*Definition of an Opinion as a Quintuple* (lecture 10-2 p. 17): $(e_i, a_{jl}, so_{ijkl}, h_j, t_k)$, that is
the **entity**, its **aspect**, the **sentiment** on that aspect, the opinion **holder**, and the **time**.
Below are the opinions in the lecture's iPhone review (p. 18), written out by hand.

In [ ]:
post = ('I bought an iPhone a few days ago. It is such a nice phone. The touch screen is really cool. '
        'The voice quality is clear too. It is much better than my old Blackberry, which was a terrible phone '
        'and so difficult to type with its tiny keys. However, my mother was mad with me as I did not tell her '
        'before I bought the phone. She also thought the phone was too expensive, ...')

quintuples = pd.DataFrame([
    ('iPhone',     'GENERAL',       'positive', 'Abc123',        '5-1-2008'),
    ('iPhone',     'touch_screen',  'positive', 'Abc123',        '5-1-2008'),
    ('iPhone',     'voice_quality', 'positive', 'Abc123',        '5-1-2008'),
    ('Blackberry', 'GENERAL',       'negative', 'Abc123',        '5-1-2008'),
    ('Blackberry', 'keys',          'negative', 'Abc123',        '5-1-2008'),
    ('iPhone',     'price',         'negative', "Abc123's mother", '5-1-2008'),
], columns=['entity', 'aspect', 'sentiment', 'holder', 'time'])

quintuples

**🤔 Quick check** One post gave six rows: two entities, four aspects, two holders. *Purpose: structure the unstructured*
(p. 19): once the opinions are rows, they can be counted, like the feature-based summary on p. 21.

In [ ]:
quintuples.groupby(['entity', 'aspect', 'sentiment']).size().unstack(fill_value=0)

### &nbsp;1-2. The simpler problem: one sentiment per document
Finding every quintuple is hard. The lecture's *simpler form of the problem* (p. 19) is **document sentiment
classification** (p. 24-25): the whole review gets one label, and star ratings give that label for free.
*4-5 stars as positive and 1-2 stars as negative. Neutral is ignored.*

In [ ]:
STAR_TO_SENTIMENT = {1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'}
train['sentiment'] = train['rating'].map(STAR_TO_SENTIMENT)
test['sentiment'] = test['rating'].map(STAR_TO_SENTIMENT)

test_pn = test[test['sentiment'] != 'neutral'].reset_index(drop=True)      # neutral is ignored
y_true = test_pn['sentiment']

print('train :', train['sentiment'].value_counts().to_dict())
print('test  :', test['sentiment'].value_counts().to_dict())
print('test reviews we evaluate on :', len(test_pn))

**🤔 Quick check** Five out of six reviews are positive. Keep this in mind when you read an accuracy below:
saying "positive" for every review already gives $1{,}124 / 1{,}352 = 0.831$.

## &nbsp;2. Dictionary-based approach
*Utilization of Sentiment Lexicon* (lecture 11 p. 4): a lexicon is a list of words with a polarity. Using the words that make
up a document, the sentiment of the document is decided **without any training**.

### &nbsp;2-1. A public lexicon: the opinion lexicon of Hu and Liu
One of the *popular sentiment lexicon databases for English* (p. 13).

In [ ]:
POS_WORDS = set(opinion_lexicon.positive())
NEG_WORDS = set(opinion_lexicon.negative())
print('positive words :', len(POS_WORDS), '  negative words :', len(NEG_WORDS))
print('positive e.g. :', sorted(POS_WORDS)[100:110])
print('negative e.g. :', sorted(NEG_WORDS)[100:110])

hotel_words = ['clean', 'spacious', 'quiet', 'friendly', 'dirty', 'noisy', 'rude', 'expensive', 'small', 'tiny', 'old']
pd.DataFrame({'in positive list': [w in POS_WORDS for w in hotel_words],
              'in negative list': [w in NEG_WORDS for w in hotel_words]}, index=hotel_words).T

**🤔 Quick check** *small*, *tiny* and *old* are in neither list, although a small, old room is a complaint in a hotel review.
*Many of them are context dependent, not just application domain dependent* (p. 3).

### &nbsp;2-2. Count the opinion words
The procedure of p. 44: **count the (unique) positive and negative words** of a text and assign it to the side with more words.
When the two counts are equal we say *negative* (our own choice, not a general rule), because a review with as much complaint as praise is rarely a 4-5 star review.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z][a-z'-]*", text.lower())


def count_opinion_words(text, pos_words, neg_words):
    words = set(tokenize(text))                       # unique words of the text
    return len(words & pos_words), len(words & neg_words)


def lexicon_classify(text, pos_words, neg_words):
    n_pos, n_neg = count_opinion_words(text, pos_words, neg_words)
    return 'positive' if n_pos > n_neg else 'negative'


examples = ['Honda Accords and Toyota Camrys are nice sedans',                                  # lecture 10-2 p. 4
            'Honda Accords and Toyota Camrys are nice sedans, but hardly the best car on the road',
            'The voice of this phone is clear',                                                 # lecture 10-2 p. 20
            'I love this phone',
            'I am so surprised to see you']
for s in examples:
    words = set(tokenize(s))
    print(f'{lexicon_classify(s, POS_WORDS, NEG_WORDS):9}', s)
    print(f'{"":9} + {sorted(words & POS_WORDS)}   - {sorted(words & NEG_WORDS)}')

**🤔 Quick check** *Sentiment analysis can be tricky* (10-2 p. 4): the second sedan sentence is a complaint, but the lexicon
only sees *nice* and *best*. *hardly* is in neither list and turns nothing around. The last sentence expresses an emotion
but no opinion (p. 20): it has no opinion word on either side. Our tie rule still returns *negative*, so this is a
mistake of the classifier, not a negative opinion.

### &nbsp;2-3. All 1,352 test reviews

In [ ]:
counts = np.array([count_opinion_words(t, POS_WORDS, NEG_WORDS) for t in test_pn['text']])
pred_lexicon = np.where(counts[:, 0] > counts[:, 1], 'positive', 'negative')


def report(name, y_true, y_pred):
    f1_neg, f1_pos = f1_score(y_true, y_pred, labels=['negative', 'positive'], average=None)
    print(f'{name:28} accuracy {accuracy_score(y_true, y_pred):.3f}   F1 negative {f1_neg:.3f}   F1 positive {f1_pos:.3f}')


def confusion(y_true, y_pred):
    # rows = actual, columns = prediction, as in lecture 11 p. 45
    return pd.crosstab(pd.Series(y_true, name='actual'), pd.Series(y_pred, name='prediction'))


report('Opinion lexicon (count)', y_true, pred_lexicon)
print('average opinion words per review : +%.1f  -%.1f' % tuple(counts.mean(axis=0)))
print('ties (n_pos == n_neg) :', (counts[:, 0] == counts[:, 1]).sum())
confusion(y_true, pred_lexicon)

**🤔 Quick check** Compare with the news experiment of lecture 11 p. 45: accuracy 90.28 %, F1 94 % for the positive class and
**69 %** for the negative class. Our numbers follow the same pattern. The large class is easy and the small negative class
is where the lexicon struggles: 86 negative reviews were called positive.

### &nbsp;2-4. Where the lexicon goes wrong
*Example: Misidentifying a true positive as a negative* and *Mistaking a true negative as a positive* (p. 47, 49).

In [ ]:
def show(rows, n=2):
    for _, r in rows.head(n).iterrows():
        words = set(tokenize(r['text']))
        print(f"[{r['rating']} stars]  + {sorted(words & POS_WORDS)}")
        print(f"{'':10} - {sorted(words & NEG_WORDS)}")
        print(f"{'':10} {r['text'][:260]} ...\n")


errors = test_pn.assign(pred=pred_lexicon)
print('=== true positive, predicted negative ===')
show(errors[(errors['sentiment'] == 'positive') & (errors['pred'] == 'negative')])
print('=== true negative, predicted positive ===')
show(errors[(errors['sentiment'] == 'negative') & (errors['pred'] == 'positive')])

**🤔 Quick check** Happy guests also write about what went wrong, and unhappy guests are polite about the staff or the
location before they complain. In the first review *not mind ... shabby lobby* and *not let ... scare* are praise, but the
lexicon cannot see that *not* turns them around. Every word gets the same single vote, whatever its order or importance.

### &nbsp;2-5. Growing a lexicon from WordNet
*Dictionary-based approaches* (p. 11): *use WordNet's synsets to acquire opinion words, starting with a small seed set*.
The synonyms of a positive word are added as positive, its antonyms as negative, and the other way round.

In [ ]:
def expand_with_wordnet(pos, neg):
    new_pos, new_neg = set(pos), set(neg)
    for words, same, opposite in [(pos, new_pos, new_neg), (neg, new_neg, new_pos)]:
        for word in words:
            for synset in wn.synsets(word, pos=wn.ADJ):       # adjective senses of the word
                for lemma in synset.lemmas():
                    same.add(lemma.name())                    # synonym  -> same polarity
                    for antonym in lemma.antonyms():
                        opposite.add(antonym.name())          # antonym  -> opposite polarity
    return new_pos, new_neg


pos, neg = {'good', 'clean', 'quiet'}, {'bad', 'dirty', 'noisy'}
for step in [1, 2, 3]:
    pos, neg = expand_with_wordnet(pos, neg)
    both = pos & neg
    print(f'step {step} : positive {len(pos):4}   negative {len(neg):4}   in both lists {len(both):3}  {sorted(both)[:6]}')
    if step == 1:
        print('         positive e.g.', sorted(pos)[:14])
        print('         negative e.g.', sorted(neg)[:14])

**🤔 Quick check** Six seeds become hundreds of words, but the meaning drifts, because every adjective sense of a word is used. After step 1 *blank* and *white* are positive
(they share the sense "not written on" with *clean*), *big* is negative (*bad* as in "a bad cold" means "very intense")
and *active* is negative (the antonym of *quiet* for the sun). From step 2 some words are in both lists.
*Usually do not give domain or context dependent meanings* (p. 11). SentiWordNet (p. 12) shows the same problem as numbers:
one score per **sense**, and the lexicon does not know which sense a hotel review means.

In [ ]:
for word in ['cheap', 'small', 'cool']:
    for s in list(swn.senti_synsets(word, 'a'))[:3]:
        print(f'{word:6} {s.synset.name():14} pos {s.pos_score():.3f}  neg {s.neg_score():.3f}   {s.synset.definition()[:60]}')
    print()

## &nbsp;3. Corpus-based approach : a sentiment dictionary from the ratings
*Building up a sentiment dictionary based on movie review data* (lecture 11 p. 17-20). Our corpus comes with ratings,
so we can ask every word: **are the reviews that contain it rated higher or lower than the reviews that do not?**

$$Score(w) = E(w) = \text{average rating of the reviews with } w \qquad\qquad
T_w = \frac{E(w) - E(W)}{\sqrt{\dfrac{s_w^2}{n(w)} + \dfrac{s_W^2}{n(W)}}}$$

Here $W$ is the set of reviews **without** the word, $s^2$ a variance and $n$ a number of reviews. The word is
**positive** if $T_w > t_{(\alpha; v)}$, **negative** if $T_w < -t_{(\alpha; v)}$, and **neutral** otherwise, where $v$ is
the degree of freedom of p. 19 (Welch's t-test). A positive $T_w$ means that the reviews with the word are
rated higher, as for the positive words in the table of p. 20.

### &nbsp;3-1. One word by hand

In [ ]:
ratings = train['rating'].to_numpy(dtype=float)        # all 6,000 training reviews, 3 stars included


def t_test_word(word, alpha=0.05):
    has = train['text'].map(lambda t: word in set(tokenize(t))).to_numpy()
    with_w, without_w = ratings[has], ratings[~has]
    a = with_w.var(ddof=1) / len(with_w)               # s_w^2 / n(w)
    b = without_w.var(ddof=1) / len(without_w)         # s_W^2 / n(W)
    T = (with_w.mean() - without_w.mean()) / np.sqrt(a + b)
    v = (a + b) ** 2 / (a ** 2 / (len(with_w) - 1) + b ** 2 / (len(without_w) - 1))
    t_crit = stats.t.ppf(1 - alpha / 2, v)             # two-sided, alpha = 0.05
    diag = 'Positive' if T > t_crit else 'Negative' if T < -t_crit else 'Neutral'
    print(f'{word:10} E(w) {with_w.mean():.2f}  n(w) {len(with_w):5}   E(W) {without_w.mean():.2f}  n(W) {len(without_w):5}'
          f'   T {T:7.2f}  t {t_crit:.2f}  -> {diag}')


for word in ['excellent', 'great', 'good', 'breakfast', 'dirty']:
    t_test_word(word)

**🤔 Quick check** *good* is in the positive list of §2, but reviews that say *good* are rated **lower** than the others
(3.95 vs 4.02). In a corpus where most guests write *great* or *excellent*, *good* sounds lukewarm.

### &nbsp;3-2. Every word at once
The same test for every word that appears in at least 20 training reviews. A binary document-term matrix (Week 4) gives
all the sums at once: `X.T @ ratings` is the sum of the ratings of the reviews that contain each word.

In [ ]:
def corpus_dictionary(texts, ratings, min_df=20, alpha=0.05):
    vec = CountVectorizer(binary=True, min_df=min_df, token_pattern=r"[a-z][a-z'-]*")
    X = vec.fit_transform(texts)                        # 1 if the review contains the word
    n_w = np.asarray(X.sum(axis=0)).ravel()             # n(w)
    n_W = len(ratings) - n_w                            # n(W)
    sum_w = X.T @ ratings
    sq_w = X.T @ ratings ** 2
    mean_w = sum_w / n_w
    mean_W = (ratings.sum() - sum_w) / n_W
    var_w = (sq_w - n_w * mean_w ** 2) / (n_w - 1)
    var_W = ((ratings ** 2).sum() - sq_w - n_W * mean_W ** 2) / (n_W - 1)
    a, b = var_w / n_w, var_W / n_W
    T = (mean_w - mean_W) / np.sqrt(a + b)
    v = (a + b) ** 2 / (a ** 2 / (n_w - 1) + b ** 2 / (n_W - 1))
    t_crit = stats.t.ppf(1 - alpha / 2, v)
    diag = np.where(T > t_crit, 'Positive', np.where(T < -t_crit, 'Negative', 'Neutral'))
    return pd.DataFrame({'Avg': mean_w, 'Stdev': np.sqrt(var_w), 'Count': n_w, 'T': T,
                         't-dist': t_crit, 'Diag': diag}, index=vec.get_feature_names_out())


table = corpus_dictionary(train['text'], ratings)
print(len(table), 'words :', table['Diag'].value_counts().to_dict())
pd.concat([table.nlargest(8, 'T'), table[table['Diag'] == 'Neutral'].sample(4, random_state=0),
           table.nsmallest(8, 'T')]).round(2)

**🤔 Quick check** The layout of the table on p. 20. *loved*, *excellent*, *fantastic* on top, *dirty*, *terrible*, *rude* at
the bottom, and in between words whose reviews are rated like any other review. *not* and *no* are strongly negative,
and so is *told* (*"we were told ..."* starts many complaints): no general-purpose lexicon lists these words.
A large or small T describes how a word is **associated** with the ratings. A negative T means that reviews containing
the word tend to have lower ratings than reviews without it; it does not mean that the word itself has a negative meaning.

### &nbsp;3-3. Domain-dependent orientations
*Corpus-based approaches: can find domain dependent orientations* (p. 11). Put the two dictionaries side by side.

In [ ]:
corpus_pos = set(table.index[table['Diag'] == 'Positive'])
corpus_neg = set(table.index[table['Diag'] == 'Negative'])

flipped = sorted((corpus_neg & POS_WORDS) | (corpus_pos & NEG_WORDS))
print('opinion lexicon says positive, hotel ratings say negative :', sorted(corpus_neg & POS_WORDS))
print('opinion lexicon says negative, hotel ratings say positive :', sorted(corpus_pos & NEG_WORDS))
table.loc[['good', 'fine', 'decent', 'pretty', 'pricey', 'downside', 'worry']].round(2)

**🤔 Quick check** *good*, *fine* and *decent* lose to the 4-5 star vocabulary, while *pricey*, *downside* and *worry* come
from happy guests (*"a bit pricey but worth it"*, *"only downside"*, *"don't worry"*). The ratings, not a dictionary
editor, decide the polarity, and that is exactly what a hotel company would want.

### &nbsp;3-4. 🔧 The corpus dictionary as a classifier
Use the corpus dictionary with the counting rule of §2-2 on the same 1,352 test reviews.
Run the cell once, then **change `ALPHA = 0.05` to `ALPHA = 0.001`** and run it again: a stricter test keeps fewer words.
Set it back to `0.05` afterwards if you want the summary below to match the slides.

In [ ]:
ALPHA = 0.05                                    # 🔧 change to 0.001 and run again

table_a = corpus_dictionary(train['text'], ratings, alpha=ALPHA)
corpus_pos = set(table_a.index[table_a['Diag'] == 'Positive'])
corpus_neg = set(table_a.index[table_a['Diag'] == 'Negative'])
pred_corpus = [lexicon_classify(t, corpus_pos, corpus_neg) for t in test_pn['text']]

print(f'alpha = {ALPHA} : {len(corpus_pos)} positive, {len(corpus_neg)} negative words')
report('Opinion lexicon (count)', y_true, pred_lexicon)
report('Corpus dictionary (count)', y_true, pred_corpus)

**🤔 Quick check** With either $\alpha$, the domain dictionary classifies **worse** than the general lexicon (0.798 vs 0.925 at
$\alpha = 0.05$). Its words are right on average, but the counting rule gives *not* (in 71 % of the training reviews) the same single vote
as *filthy*. A long positive review with a few *not* and *no* is outvoted. A word needs a **weight**, and learning that
weight from the ratings is the machine learning-based approach of Notebook 2.

## &nbsp;Summary

In [ ]:
summary = pd.DataFrame([
    ('Always "positive" (baseline)', 'nothing', (y_true == 'positive').mean()),
    ('Opinion lexicon (count)', 'a public word list', accuracy_score(y_true, pred_lexicon)),
    (f'Corpus dictionary (alpha {ALPHA})', 'ratings of 6,000 reviews', accuracy_score(y_true, pred_corpus)),
], columns=['method', 'what it needs', 'accuracy']).round(3)
summary

| Lecture | Code | What we saw |
|---|---|---|
| 10-2 p. 17-19 | `quintuples` | an opinion becomes a row: entity, aspect, sentiment, holder, time |
| 10-2 p. 24-25 | `STAR_TO_SENTIMENT` | document-level labels from ratings, neutral ignored |
| 11 p. 4, 44-45 | `lexicon_classify` | count positive vs negative words: accuracy 0.925, F1 negative 0.736 |
| 11 p. 11-12 | `expand_with_wordnet`, `swn` | seeds grow quickly but the meaning drifts, one score per sense |
| 11 p. 17-20 | `corpus_dictionary` | a t-test per word: *good* is negative and *downside* is positive for hotels |

**Next: Notebook 2** learns a weight for every word from labeled reviews (classifiers and Elastic Net), and spreads
the sentiment of a few seed words to a whole vocabulary (semi-supervised learning).